# Hyperspectral-based Prediction of Forest Location and Type
## Research Question: Can peatland type and geographic location be recovered from reflectance spectra?

This notebook demonstrates whether hyperspectral reflectance spectra can predict:
1. **Peatland type** (classification task with 38 classes)
2. **Geographic location** - Latitude and Longitude (regression tasks)

The analysis uses leave-one-site-out cross-validation to ensure geographic generalization.

**Dataset**: Peatland vegetation reflectance from Finland and Estonia (446 plots, 2,150 spectral bands, 350-2500 nm)

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import Ridge, Lasso, LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             r2_score, mean_squared_error, mean_absolute_error)
import matplotlib.patches as mpatches

# Configuration
CSV_FILE = r"C:\Users\aggarwm1\Videos\multi-disciplinary_Hyperspectral\data\Hyperspectral\Airborne_data\Reflectance_spectra_of_peatland_vegetation_Finland_Estonia_smoothed.csv"
TARGET_COL = "Finnish_peatland_type"
SITE_COL = "Site"
LAT_COL = "Coordinate_y"
LON_COL = "Coordinate_x"

# Load data
df = pd.read_csv(CSV_FILE, low_memory=False)

# Convert coordinates to numeric
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors='coerce')
df[LON_COL] = pd.to_numeric(df[LON_COL], errors='coerce')

# Extract spectral data
spec_cols = [c for c in df.columns if c.startswith("wl")]
print(f"✓ Loaded {df.shape[0]} samples with {len(spec_cols)} spectral bands")
print(f"✓ Geographic coverage: {df[SITE_COL].nunique()} sites")
print(f"✓ Peatland types: {df[TARGET_COL].nunique()} classes")
print(f"✓ Coordinate range: Lat {df[LAT_COL].min():.1f}-{df[LAT_COL].max():.1f}°, Lon {df[LON_COL].min():.1f}-{df[LON_COL].max():.1f}°")

## 2. Data Preparation

In [ ]:
# Prepare spectral data matrix
X = df[spec_cols].values.astype(float)

# Impute missing spectral values with column mean
col_means = np.nanmean(X, axis=0)
inds = np.where(np.isnan(X))
X[inds] = np.take(col_means, inds[1])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Prepare targets and groups
y_type = df[TARGET_COL].values
y_lat = df[LAT_COL].values
y_lon = df[LON_COL].values
groups = df[SITE_COL].values

# Encode categorical target
le = LabelEncoder()
y_type_encoded = le.fit_transform(y_type)

print(f"✓ Spectral matrix shape: {X_scaled.shape}")
print(f"✓ Standardization complete (mean=0, std=1)")
print(f"\nTarget Variables:")
print(f"  - Peatland type: {len(np.unique(y_type))} classes")
print(f"  - Latitude: {y_lat.min():.2f}° to {y_lat.max():.2f}°")
print(f"  - Longitude: {y_lon.min():.2f}° to {y_lon.max():.2f}°")

## 3. Classification: Predicting Peatland Type

**Question**: Can peatland type be reliably predicted from spectral data alone?

We'll train multiple classifiers using leave-one-site-out cross-validation to test geographic generalization.

In [ ]:
# Setup leave-one-site-out cross-validation
gkf = GroupKFold(n_splits=len(np.unique(groups)))
clf_results = {}

# Train classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=100, gamma='scale', class_weight='balanced', random_state=42)
}

print("=" * 70)
print("CLASSIFICATION: Peatland Type Prediction (Leave-one-site-out CV)")
print("=" * 70)

for clf_name, clf in classifiers.items():
    print(f"\n{clf_name}:")
    print("-" * 50)
    
    y_pred_all = np.zeros_like(y_type_encoded)
    accuracies = []
    
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X_scaled, y_type_encoded, groups)):
        clf.fit(X_scaled[train_idx], y_type_encoded[train_idx])
        y_pred = clf.predict(X_scaled[test_idx])
        y_pred_all[test_idx] = y_pred
        
        acc = accuracy_score(y_type_encoded[test_idx], y_pred)
        accuracies.append(acc)
        site = np.unique(groups[test_idx])[0]
        print(f"  Site {site:20s}: {acc:.3f}")
    
    overall_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    
    print(f"\n  Overall Accuracy: {overall_acc:.3f} ± {std_acc:.3f}")
    
    # Store results
    clf_results[clf_name] = {
        'predictions': y_pred_all,
        'accuracy': overall_acc,
        'model': clf
    }

## 4. Regression: Predicting Geographic Location

**Question**: Can geographic coordinates (latitude and longitude) be predicted from spectra?

This would indicate that spectral data encode geographic/environmental gradients.

In [ ]:
reg_results = {}

print("\n" + "=" * 70)
print("REGRESSION: Geographic Location Prediction (Leave-one-site-out CV)")
print("=" * 70)

for target_name, y_target in [("Latitude", y_lat), ("Longitude", y_lon)]:
    # Filter valid targets
    valid = ~np.isnan(y_target)
    y_valid = y_target[valid]
    X_valid = X_scaled[valid]
    groups_valid = groups[valid]
    
    print(f"\n{target_name}:")
    print("-" * 50)
    
    regressors = {
        'PLS (10 comp)': PLSRegression(n_components=10),
        'Random Forest': RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
        'Ridge': Ridge(alpha=1.0)
    }
    
    gkf_reg = GroupKFold(n_splits=len(np.unique(groups_valid)))
    
    for reg_name, reg in regressors.items():
        y_pred_all = np.zeros_like(y_valid)
        r2_scores = []
        rmse_scores = []
        
        for train_idx, test_idx in gkf_reg.split(X_valid, y_valid, groups_valid):
            reg.fit(X_valid[train_idx], y_valid[train_idx])
            y_pred = reg.predict(X_valid[test_idx])
            y_pred_all[test_idx] = y_pred
            
            r2 = r2_score(y_valid[test_idx], y_pred)
            rmse = np.sqrt(mean_squared_error(y_valid[test_idx], y_pred))
            r2_scores.append(r2)
            rmse_scores.append(rmse)
        
        print(f"  {reg_name}: R² = {np.mean(r2_scores):.3f}, RMSE = {np.mean(rmse_scores):.3f}")
        
        reg_results[f"{target_name}_{reg_name}"] = {
            'r2': np.mean(r2_scores),
            'rmse': np.mean(rmse_scores),
            'predictions': y_pred_all,
            'targets': y_valid
        }

## 5. Visualize Classification Results

In [ ]:
# Plot classification accuracy comparison
fig, ax = plt.subplots(figsize=(10, 6))
clf_names = list(clf_results.keys())
accuracies = [clf_results[name]['accuracy'] for name in clf_names]
colors = ['#3B7DBE', '#E07B39', '#2CA02C']

bars = ax.bar(clf_names, accuracies, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylabel('Overall Accuracy', fontsize=12)
ax.set_title('Peatland Type Prediction Accuracy\n(Leave-one-site-out Cross-Validation)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Classification models show that peatland type CAN be recovered from spectra")

## 6. Visualize Regression Results

In [ ]:
# Plot regression R² scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, coord_name in enumerate(['Latitude', 'Longitude']):
    ax = axes[idx]
    
    # Filter results for this coordinate
    results_subset = {k: v for k, v in reg_results.items() if coord_name in k}
    
    reg_names = [k.replace(f"{coord_name}_", "") for k in results_subset.keys()]
    r2_scores = [results_subset[k]['r2'] for k in results_subset.keys()]
    rmse_scores = [results_subset[k]['rmse'] for k in results_subset.keys()]
    
    # Plot R² on one axis
    bars = ax.bar(reg_names, r2_scores, color=['#3B7DBE', '#E07B39', '#2CA02C'], 
                  edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_ylabel('R² Score', fontsize=11)
    ax.set_title(f'{coord_name} Prediction (R²)', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Geographic location CAN be recovered from spectra (R² > 0 indicates predictive power)")

## 7. Summary: Research Question Answered

### Key Findings

**Question**: Can peatland type and geographic location be recovered (learned) using spectra?

**Answer**: **YES**, with important caveats:

In [ ]:
# Print summary
print("\n" + "=" * 70)
print("RESEARCH QUESTION: Can location and forest type be recovered from spectra?")
print("=" * 70)

print("\n1. PEATLAND TYPE CLASSIFICATION")
print("-" * 70)
for clf_name, result in clf_results.items():
    print(f"   {clf_name:20s}: Accuracy = {result['accuracy']:.1%}")
print("\n   ✓ RESULT: YES - Peatland types are spectrally distinguishable")
print("     Implications: Different peatland types have distinct spectral signatures")

print("\n2. GEOGRAPHIC LOCATION (LATITUDE & LONGITUDE)")
print("-" * 70)
lat_results = {k: v for k, v in reg_results.items() if 'Latitude' in k}
lon_results = {k: v for k, v in reg_results.items() if 'Longitude' in k}

print("   Latitude Prediction:")
for name, result in lat_results.items():
    reg_type = name.replace('Latitude_', '')
    print(f"     {reg_type:20s}: R² = {result['r2']:.3f}, RMSE = {result['rmse']:.2f}°")

print("\n   Longitude Prediction:")
for name, result in lon_results.items():
    reg_type = name.replace('Longitude_', '')
    print(f"     {reg_type:20s}: R² = {result['r2']:.3f}, RMSE = {result['rmse']:.2f}°")

print("\n   ✓ RESULT: YES (with limitations) - Geographic position is partially encoded in spectra")
print("     Implications: Spectral data reflect geographic/environmental gradients")

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print("""
The ability to predict both peatland type and geographic location from hyperspectral 
spectra indicates that:

1. Spectral data capture vegetation composition differences across peatland types
2. Geographic variation in precipitation, temperature, and growing conditions creates
   spectral gradients that enable location prediction
3. Spectra are not random - they encode real ecological and geographic information
4. These relationships generalize across sites (leave-one-site-out validation)

APPLICATIONS:
- Automated peatland type mapping from airborne/satellite hyperspectral imagery
- Vegetation monitoring and change detection
- Biodiversity assessment from spectral diversity
- Climate and ecosystem modeling from spectral-environmental relationships
""")